# Parte 2 — Ejecución 100% en Notebook (robusta)
Corre la Parte 2 detectando automáticamente la raíz del proyecto, verificando el dataset,
ejecutando el entrenamiento y generando las figuras.


In [ ]:
# (Opcional) Instalar dependencias desde aquí
import sys, subprocess
from pathlib import Path
REQ = (Path.cwd().parent / 'requerimientos.txt')
print('Intérprete:', sys.executable)
print('requerimientos.txt:', REQ)
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REQ)])  # descomenta si lo necesitas


In [ ]:
# 1) Detectar raíz del proyecto y preparar rutas
from pathlib import Path
def detectar_raiz(p: Path) -> Path:
    cur = p.resolve()
    while True:
        if (cur / 'configuraciones').exists() and (cur / 'aplicacion').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return p
CWD = Path().resolve()
RAIZ = detectar_raiz(CWD)
CFG = RAIZ / 'configuraciones' / 'parte2.json'
print('Carpeta actual:', CWD)
print('Raíz detectada:', RAIZ)
print('Config:', CFG)
assert CFG.exists(), f'No se encontró configuración en {CFG}'


In [ ]:
# 2) Verificar dataset y mostrar resumen de configuración
import json
cfg = json.load(open(CFG, 'r', encoding='utf-8'))
print(json.dumps(cfg, ensure_ascii=False, indent=2))
from pathlib import Path
cfg_dir = CFG.parent
ruta_datos = cfg['datos']['ruta']
ds_abs = (cfg_dir / ruta_datos).resolve() if not Path(ruta_datos).is_absolute() else Path(ruta_datos)
print('Dataset resuelto:', ds_abs)
assert ds_abs.exists(), f'No se encontró el dataset en {ds_abs}. Corrige "datos.ruta".'


In [ ]:
# 3) Ejecutar entrenamiento
import sys
sys.path.append(str(RAIZ))
from aplicacion.entrenamiento import ejecutar
run_dir = ejecutar(str(CFG))
print('\nArtefactos generados en:', run_dir)


In [ ]:
# 4) Generar figuras automáticamente
from herramientas.graficos import curvas_entrenamiento, matrices_confusion
from pathlib import Path
RUN = Path(run_dir)
if not RUN.exists():
    base = RAIZ / 'resultados'
    cand = sorted([d for d in base.iterdir() if d.is_dir() and d.name.startswith('ejecucion-')])
    RUN = cand[-1] if cand else base
print('Usando carpeta para gráficas:', RUN)
curvas_entrenamiento(str(RUN))
matrices_confusion(str(RUN))
print('Listo: figuras guardadas en', RUN)
